# DR-ME Benchmarks

In [ ]:
!git clone https://github.com/houssamzenati/interpretable-dr-me-test code/

import sys
sys.path.insert(0, "code/interpretable-dr-me-test")

In [ ]:
import numpy as np

import exp1_calibration_dr_me_plugin as exp1
from exp3_hd_whitening_ablation import fast_full_greedy_select, make_observed_dictionary

from causal_medmnist import Scenario
from causal_medmnist.datasets import REGISTRY

In [ ]:
n_samples = 1000
num_repetitions = 200
J = 5
M = 250
bandwidth_scale = 0.70
tau = 1e-4
ridge = 1e-8


def generate(scenario, n, rng, split):
    sample = scenario.generate(n, seed=int(rng.integers(0, 2**32)), split=split, replace=True)
    return sample.X, sample.A, sample.Y.reshape(n, -1).astype(float)


def run_test(dataset, n_samples, scale, rng):
    scenario = Scenario(dataset, effect_strength=scale)
    rejections = []

    for _ in range(num_repetitions):
        X, A, Y = generate(scenario, n_samples, rng, "train")

        perm = rng.permutation(n_samples)
        n_third = n_samples // 3
        idx_eta, idx_tr, idx_te = perm[:n_third], perm[n_third:2 * n_third], perm[2 * n_third:]

        C, _ = make_observed_dictionary(Y[idx_tr], idx_tr, M, rng)
        ell_y = max(bandwidth_scale * exp1.median_bandwidth_y(Y[np.concatenate([idx_eta, idx_tr])], rng), 1e-3)

        nuis = exp1.PluginNuisance(ell_y=ell_y).fit(X[idx_eta], A[idx_eta], Y[idx_eta], C)

        Z_train = exp1.pseudo_feature_matrix("dr", X[idx_tr], A[idx_tr], Y[idx_tr], C, ell_y, nuis)
        Z_test = exp1.pseudo_feature_matrix("dr", X[idx_te], A[idx_te], Y[idx_te], C, ell_y, nuis)

        cols = fast_full_greedy_select(Z_train, J, tau)
        _, pval = exp1.hotelling_pvalue(Z_test, cols, ridge=ridge)
        rejections.append(float(pval < 0.05))

    return np.mean(rejections)


def run(scale, rng):
    for dataset in sorted(REGISTRY):
        result = run_test(dataset, n_samples, scale, rng)
        print(f"[{dataset}][N={n_samples}][scale={scale}] {result}")

In [ ]:
rng = np.random.default_rng(0)

run(scale=0.6, rng=rng)
run(scale=0.0, rng=rng)